# 09 - Results Analysis

Este notebook sintetiza os principais resultados obtidos até agora:
- escolha da segmentação;
- baselines finais;
- importance de features;
- seleção de features para utilidade;
- trade-off entre utilidade e linkability;
- robustness da linkability;
- transformações de privacidade.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import FEATURE_SETS_DIR, FINAL_SEGMENT_FEATURES_DIR, OUTPUTS_TABLES_DIR
from modeling import compute_linkability_feature_importance, compute_utility_feature_importance

## Final Dataset Summary

In [ ]:
manifest = json.loads((FINAL_SEGMENT_FEATURES_DIR / "manifest.json").read_text(encoding="utf-8"))
errors_path = FINAL_SEGMENT_FEATURES_DIR / "errors.csv"
errors_df = pd.read_csv(errors_path) if errors_path.exists() and errors_path.stat().st_size > 0 else pd.DataFrame()

pd.DataFrame(
    {
        "metric": [
            "record_count",
            "window_sec",
            "step_sec",
            "total_segments",
            "error_count",
            "chunk_count",
        ],
        "value": [
            manifest["record_count"],
            manifest["window_sec"],
            manifest["step_sec"],
            manifest["total_segments"],
            manifest.get("error_count", len(errors_df)),
            len(manifest["chunks"]),
        ],
    }
)

## Segmentation Selection Summary

Resumo dos dois candidatos finais avaliados no estudo de segmentação.

In [ ]:
segmentation_selection_df = pd.DataFrame(
    [
        {
            "config": "w2_o0p5",
            "window_sec": 2.0,
            "step_sec": 1.0,
            "utility_logreg_f1": 0.723567,
            "utility_logreg_balanced_accuracy": 0.854232,
            "utility_xgb_f1": 0.723949,
            "utility_xgb_roc_auc": 0.939952,
            "linkability_logreg_roc_auc": 0.996212,
            "linkability_xgb_roc_auc": 0.998898,
            "decision": "selected_main",
        },
        {
            "config": "w3_o0p5",
            "window_sec": 3.0,
            "step_sec": 1.5,
            "utility_logreg_f1": 0.737304,
            "utility_logreg_balanced_accuracy": 0.863631,
            "utility_xgb_f1": 0.754435,
            "utility_xgb_roc_auc": 0.950988,
            "linkability_logreg_roc_auc": 0.999280,
            "linkability_xgb_roc_auc": 0.999357,
            "decision": "higher_utility_comparison",
        },
    ]
)
segmentation_selection_df

## Final Baselines

Resultados finais das baselines no dataset final selecionado.

In [ ]:
baseline_results_df = pd.DataFrame(
    [
        {
            "task": "utility",
            "model": "LogisticRegression",
            "f1_score": 0.678900,
            "balanced_accuracy": 0.831770,
            "roc_auc": 0.907671,
            "pr_auc": 0.712424,
            "status": "selected_baseline",
        },
        {
            "task": "utility",
            "model": "XGBoost",
            "f1_score": 0.675260,
            "balanced_accuracy": 0.775427,
            "roc_auc": 0.925902,
            "pr_auc": 0.779992,
            "status": "comparison",
        },
        {
            "task": "linkability",
            "model": "LogisticRegression",
            "f1_score": 0.975219,
            "balanced_accuracy": None,
            "roc_auc": 0.996190,
            "pr_auc": 0.996639,
            "status": "comparison",
        },
        {
            "task": "linkability",
            "model": "XGBoost",
            "f1_score": 0.978809,
            "balanced_accuracy": None,
            "roc_auc": 0.998343,
            "pr_auc": 0.998426,
            "status": "selected_baseline",
        },
    ]
)
baseline_results_df

## Utility Feature Selection

Resumo dos experimentos `top-k` e da escolha do subset `top-150`.

In [ ]:
utility_feature_selection_summary = pd.read_csv(FEATURE_SETS_DIR / "utility_feature_selection_summary.csv")
utility_feature_selection_summary

## Full 208 vs Utility Top-150

Comparação usando exatamente os mesmos segmentos para utilidade e linkability.

In [ ]:
same_data_tradeoff_df = pd.DataFrame(
    [
        {
            "feature_set": "full_208",
            "utility_f1": 0.678900,
            "utility_balanced_accuracy": 0.831770,
            "utility_roc_auc": 0.907671,
            "linkability_logreg_roc_auc": 0.996190,
            "linkability_xgb_roc_auc": 0.998343,
        },
        {
            "feature_set": "utility_top150",
            "utility_f1": 0.678106,
            "utility_balanced_accuracy": 0.831456,
            "utility_roc_auc": 0.906530,
            "linkability_logreg_roc_auc": 0.993848,
            "linkability_xgb_roc_auc": 0.997051,
        },
    ]
)
same_data_tradeoff_df

## Progressive Feature Removal from Top-150

Remoção das features mais importantes para linkability a partir do subset `top-150`.

In [ ]:
feature_removal_tradeoff_df = pd.DataFrame(
    [
        {"n_removed": 0, "utility_f1": 0.678106, "utility_balanced_accuracy": 0.831456, "linkability_roc_auc": 0.997051},
        {"n_removed": 10, "utility_f1": 0.676987, "utility_balanced_accuracy": 0.830770, "linkability_roc_auc": 0.996015},
        {"n_removed": 20, "utility_f1": 0.665033, "utility_balanced_accuracy": 0.823306, "linkability_roc_auc": 0.995232},
        {"n_removed": 30, "utility_f1": 0.653883, "utility_balanced_accuracy": 0.815811, "linkability_roc_auc": 0.994811},
        {"n_removed": 40, "utility_f1": 0.646042, "utility_balanced_accuracy": 0.810015, "linkability_roc_auc": 0.993679},
        {"n_removed": 50, "utility_f1": 0.635907, "utility_balanced_accuracy": 0.802330, "linkability_roc_auc": 0.993234},
    ]
)
feature_removal_tradeoff_df

## Linkability Robustness Summary

Resumo dos checks de robustez com distância temporal, limitação de pares por paciente, baselines simples e múltiplas seeds.

In [ ]:
linkability_robustness_df = pd.DataFrame(
    [
        {"setting": "standard", "model": "LogisticRegression", "f1_score": 0.982728, "roc_auc": 0.998161},
        {"setting": "standard", "model": "XGBoost", "f1_score": 0.983903, "roc_auc": 0.999101},
        {"setting": "harder", "model": "CosineSimilarity", "f1_score": 0.829690, "roc_auc": 0.892479},
        {"setting": "harder", "model": "EuclideanDistance", "f1_score": 0.000000, "roc_auc": 0.812707},
        {"setting": "harder", "model": "LogisticRegression", "f1_score": 0.982482, "roc_auc": 0.996604},
        {"setting": "harder", "model": "XGBoost", "f1_score": 0.983229, "roc_auc": 0.998676},
        {"setting": "repeated_mean", "model": "CosineSimilarity", "f1_score": 0.818678, "roc_auc": 0.884830},
        {"setting": "repeated_mean", "model": "EuclideanDistance", "f1_score": 0.000000, "roc_auc": 0.804087},
        {"setting": "repeated_mean", "model": "LogisticRegression", "f1_score": 0.981341, "roc_auc": 0.996924},
        {"setting": "repeated_mean", "model": "XGBoost", "f1_score": 0.983195, "roc_auc": 0.998672},
    ]
)
linkability_robustness_df

## Privacy Transformations Summary

Resultados mais recentes das transformações de privacidade.

In [ ]:
privacy_transformations_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transformations_summary.csv")
privacy_transformations_df

## Feature Importance Refresh

Esta secção mantém a análise de importance no dataset final selecionado. Se necessário, limita `MAX_CHUNKS` para uma corrida mais leve.

In [ ]:
MAX_CHUNKS = 20  # set None for the full dataset if needed

chunk_files = [FINAL_SEGMENT_FEATURES_DIR / chunk["chunk_file"] for chunk in manifest["chunks"]]
if MAX_CHUNKS is not None:
    chunk_files = chunk_files[:MAX_CHUNKS]

features_df = pd.concat(
    [pd.read_csv(chunk_file, compression="gzip", low_memory=False) for chunk_file in chunk_files],
    ignore_index=True,
)

print("Chunk files loaded for importance:", len(chunk_files))
print("Features dataframe:", features_df.shape)

In [ ]:
utility_importance = compute_utility_feature_importance(
    features_df=features_df,
    model_name="LogisticRegression",
    test_size=0.2,
    random_state=42,
    permutation_scoring="average_precision",
)

utility_importance["model_based_df"].head(15)

In [ ]:
linkability_importance = compute_linkability_feature_importance(
    features_df=features_df,
    model_name="LogisticRegression",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
    max_positive_pairs_per_patient=3,
    permutation_scoring="roc_auc",
)

linkability_importance["model_based_df"].head(15)

In [ ]:
utility_top = utility_importance["model_based_df"].head(20).copy()
utility_top["task"] = "utility"

linkability_top = linkability_importance["model_based_df"].head(20).copy()
linkability_top["task"] = "linkability"

comparison_df = pd.concat([utility_top, linkability_top], ignore_index=True)
comparison_df

In [ ]:
utility_feature_set = set(utility_importance["model_based_df"].head(20)["feature"])
linkability_feature_set = set(linkability_importance["model_based_df"].head(20)["feature"])

print("Overlap in top-20 model-based features:", len(utility_feature_set & linkability_feature_set))
sorted(utility_feature_set & linkability_feature_set)

## Current Takeaways

- `w2_o0p5` foi a configuração de segmentação principal escolhida.
- A baseline principal de utilidade é `LogisticRegression`.
- A baseline principal de linkability é `XGBoost`.
- `utility_top150` preserva quase toda a utilidade, mas reduz pouco a linkability.
- Remoção simples de features não reduz a linkability de forma forte.
- A linkability mantém-se muito alta mesmo com checks de robustez.
- Entre as transformações de privacidade testadas, `PCA` foi a mais promissora.
- O melhor candidato até agora é a comparação `identity` vs a melhor variante atual em `privacy_transformations_summary.csv`.